In [ ]:
"""
business_rules.py - single source of truth for business mappings, scales, data-quality ranges and POC cohort rules.
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Every script (EDA, Step 3a profiling, Step 3b cleaning, Step 3c feature engineering, Step 3d cohort builder) imports
its category lists, ranges and cohort rules from here, so the EDA and the production pipeline can never drift apart.
Values are lower-case because Silver categoricals are normalised (trim + lower) in Step 3b.
If the data dictionary or EDA Section 3A shows a different value, change it HERE only.

Change log (see decisions.md): D04 fee types, D05 DQ ranges centralised + corrected, D07 cohort rule (CORE_RULE)
shared with the EDA, D08 env override for the sample-size gate.
"""

## 1. Imports

In [ ]:
from __future__ import annotations

import os

## 2. Transactions

In [ ]:
TX_SPEND_TYPES = ("purchase", "subscription_payment", "bill_payment")   # gross spend (completed only)
TX_PAYMENT_TYPES = ("subscription_payment", "bill_payment")             # payment attempts (failure-rate denominator)
TX_REFUND_TYPES = ("refund",)
TX_REFUND_BASE_TYPES = ("purchase",)                                    # refund-rate denominator (EDA 3B)
TX_FEE_TYPES = ("fee",)                    # tracked separately from spend (EDA v5); set to () if the data has no fees
TX_SUCCESS_STATUSES = ("completed",)
TX_FAILED_STATUSES = ("failed",)
TX_REVERSED_STATUSES = ("reversed",)

## 3. Interactions

In [ ]:
INT_COMPLAINT_TYPES = ("complaint", "billing_dispute", "cancellation_request")   # adverse service contacts
INT_BILLING_DISPUTE_TYPES = ("billing_dispute",)
INT_CANCELLATION_TYPES = ("cancellation_request",)
INT_OPEN_STATUSES = ("unresolved", "escalated", "pending")    # open issue = not yet resolved
INT_ESCALATED_STATUSES = ("escalated",)                         # subset of open, reported separately
INT_RESOLVED_STATUSES = ("resolved",)
REPEAT_CONTACT_DAYS = 7

## 4. Engagements (every observed event_type must sit in exactly one of POS / NEU / NEG)

In [ ]:
ENG_POSITIVE = ("login", "feature_use", "offer_redeem", "email_click", "push_open", "sms_click", "sms_reply")
ENG_NEUTRAL = ("page_view", "offer_view", "email_open", "session_end")
ENG_NEGATIVE = ("unsubscribe", "sms_optout", "push_dismiss")     # more events is NOT always healthier
ENG_LOGIN = ("login",)
ENG_OPT_OUT = ("unsubscribe", "sms_optout")
ENG_DISMISS = ("push_dismiss",)
ENG_EMAIL_OPEN = ("email_open",)
ENG_EMAIL_CLICK = ("email_click",)
ENG_LONG_SESSION_WARN_SEC = 14_400        # > 4 h = suspicious (WARN); hard limit is RANGES['session_duration_sec']

## 5. Surveys (valid integer ratings only - never imputed)

In [ ]:
CSAT_MIN, CSAT_MAX = 1, 5
NPS_MIN, NPS_MAX = 0, 10
NPS_DETRACTOR_MAX, NPS_PASSIVE_MAX = 6, 8

## 6. Data-quality ranges (Step 3a preview + Step 3b FIX rule: out-of-range -> NULL)
One table for both steps. The previous Step 3b values (duration 540 s, session 3,700 s) contradicted their own
comments (4 h / 24 h) and nulled most interaction durations - see decisions.md D05.

In [ ]:
RANGES = {
    "age": (18, 100),
    "tenure_months": (0, 600),
    "duration_sec": (0, 14_400),            # interaction handle time: 4 hours
    "csat_score": (CSAT_MIN, CSAT_MAX),
    "nps_score": (NPS_MIN, NPS_MAX),
    "sentiment_score_gt": (-1, 1),
    "session_duration_sec": (0, 86_400),    # 24 hours (hard limit); > ENG_LONG_SESSION_WARN_SEC is a WARN
    "pages_viewed": (0, 1_000),
}

## 7. POC cohort rules (shared by EDA Step 10/12/12A and Step 3d)
A customer-month is "core" when
  HISTORY : active in the month AND months_observed >= MIN_HISTORY_MONTHS
  SOURCES : CORE_RULE evaluated on the SAME month's source flags (EDA definition)
CSAT / NPS availability is NOT required (would bias the cohort toward survey responders).

In [ ]:
MIN_HISTORY_MONTHS = 3          # months since first observed activity (incl. current) before scoring
CORE_LOOKBACK_MONTHS = 3        # rolling-window length for *_3m features and is_eligible (Step 3c)
CORE_RULE = os.getenv("CORE_RULE", "tx_plus_one")   # EDA default; switch to "any_two" if EDA 12A recommends it
CORE_RULE_DESC = {
    "tx_plus_one": "history + transactions + (engagements or interactions)",
    "any_two": "history + any 2 of transactions / engagements / interactions",
    "any_one": "history + any 1 source (reference only - not for selection)",
}
MIN_COHORT_CUSTOMERS = int(os.getenv("MIN_COHORT_CUSTOMERS", 20_000))   # sample-size gate for a candidate cohort
COHORT_DIMS = ("customer_segment", "loyalty_tier", "region", "preferred_channel", "tenure_band")
COHORT_WEIGHTS = {"value": 0.25, "coverage": 0.20, "labels": 0.15, "sample": 0.15,
                  "history": 0.10, "stability": 0.10, "homogeneity": 0.05}
TEST_SHARE = 0.20               # customer-grouped hold-out share
SPLIT_SEED = 42

## 8. Consistency checks (fail at import time, before any data is touched)

In [ ]:
assert CORE_RULE in CORE_RULE_DESC, f"CORE_RULE must be one of {list(CORE_RULE_DESC)}"
assert abs(sum(COHORT_WEIGHTS.values()) - 1) < 1e-9, "COHORT_WEIGHTS must sum to 1"
assert set(ENG_OPT_OUT) <= set(ENG_NEGATIVE), "opt-out events must be negative engagement"
assert set(ENG_DISMISS) <= set(ENG_NEGATIVE), "dismiss events must be negative engagement"
assert not (set(ENG_POSITIVE) & set(ENG_NEUTRAL) or set(ENG_POSITIVE) & set(ENG_NEGATIVE)
            or set(ENG_NEUTRAL) & set(ENG_NEGATIVE)), "engagement groups must not overlap"
assert set(INT_ESCALATED_STATUSES) <= set(INT_OPEN_STATUSES), "escalated is a subset of open"
assert not set(TX_FEE_TYPES) & set(TX_SPEND_TYPES), "fees must not be counted as spend"
assert ENG_LONG_SESSION_WARN_SEC < RANGES["session_duration_sec"][1], "WARN threshold must sit below the hard limit"

## 9. SQL helpers

In [ ]:
def tenure_band_sql(col: str) -> str:
    """Static tenure band used for cohort membership (same bands as EDA Step 9 / 12)."""
    return (f"CASE WHEN {col} IS NULL THEN 'unknown' WHEN {col} < 12 THEN '0-11m' "
            f"WHEN {col} < 24 THEN '12-23m' WHEN {col} < 48 THEN '24-47m' ELSE '48m+' END")


def sql_list(values) -> str:
    """DuckDB list literal: ['a', 'b'] (quotes escaped)."""
    return "[" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in values) + "]"


def isin(col: str, values) -> str:
    """NULL-safe membership test that always returns TRUE/FALSE (FALSE for an empty list)."""
    if not values:
        return "FALSE"
    return f"COALESCE(list_contains({sql_list(values)}, {col}), FALSE)"


def core_source_sql(tx: str, eng: str, itx: str, rule: str | None = None) -> str:
    """Source part of the core-cohort rule. Arguments are column names of 0/1 or BOOLEAN month flags,
    so the same rule runs on the EDA table (has_tx BOOLEAN) and the production table (has_txn_m INT)."""
    rule = rule or CORE_RULE
    t, e, i = (f"CAST({c} AS INT)" for c in (tx, eng, itx))
    if rule == "tx_plus_one":
        return f"({t} = 1 AND ({e} = 1 OR {i} = 1))"
    if rule == "any_two":
        return f"({t} + {e} + {i} >= 2)"
    if rule == "any_one":
        return f"({t} + {e} + {i} >= 1)"
    raise ValueError(f"unknown CORE_RULE {rule!r}")


def history_sql(active: str, months_observed: str = "months_observed") -> str:
    """History part of the core-cohort rule."""
    return f"(CAST({active} AS INT) = 1 AND {months_observed} >= {MIN_HISTORY_MONTHS})"